In [0]:
df_vbak = spark.table("sap_sd_project.bronze.vbak_raw")
display(df_vbak)

MANDT,VBELN,AUART,VKORG,VTWEG,SPART,KUNNR,AUDAT,ERDAT,WAERK,GBSTK,BSTNK
100,5000000001,ZOR,3000.0,20,1,100075,2025-03-30,2025-03-30,EGP,C,PO-94057
100,5000000002,OR,1000.0,10,1,100033,2026-03-15,2026-03-15,EGP,A,PO-69496
100,5000000003,OR,2000.0,20,1,100016,2026-07-04,2026-07-05,EGP,B,PO-75358
100,5000000004,ZOR,3000.0,10,0,100091,2025-07-06,2025-07-06,EGP,C,PO-51497
100,5000000005,ZOR,3000.0,10,0,100092,2025-10-30,2025-10-30,EGP,C,PO-99593
100,5000000006,OR,1000.0,20,0,100036,null,2025-10-20,EGP,C,PO-14269
100,5000000007,OR,2000.0,20,1,100096,2026-06-22,2026-06-22,EGP,A,PO-43081
100,5000000008,OR,2000.0,10,1,100069,2025-12-08,2025-12-09,EGP,A,PO-53734
100,5000000009,OR,1000.0,10,0,100076,2026-05-15,2026-05-15,EGP,C,PO-35036
100,5000000010,OR,2000.0,20,1,100097,2026-06-20,2026-06-20,EGP,C,PO-33777


In [0]:
df_vbak.printSchema()

root
 |-- MANDT: long (nullable = true)
 |-- VBELN: long (nullable = true)
 |-- AUART: string (nullable = true)
 |-- VKORG: double (nullable = true)
 |-- VTWEG: long (nullable = true)
 |-- SPART: long (nullable = true)
 |-- KUNNR: long (nullable = true)
 |-- AUDAT: string (nullable = true)
 |-- ERDAT: date (nullable = true)
 |-- WAERK: string (nullable = true)
 |-- GBSTK: string (nullable = true)
 |-- BSTNK: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, count, when

# Nulls in important fields
df_vbak.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["VBELN", "VKORG", "KUNNR", "AUDAT", "WAERK", "GBSTK"]
]).display()

VBELN,VKORG,KUNNR,AUDAT,WAERK,GBSTK
0,0,18,12,12,8


In [0]:
display(df_vbak.select("WAERK").distinct())
display(df_vbak.select("GBSTK").distinct())
display(df_vbak.select("VKORG").distinct())

WAERK
EGP
egp
EGP
null


GBSTK
C
A
B
B
c
complete
null


VKORG
3000.0
1000.0
2000.0


In [0]:
from pyspark.sql.functions import col, trim, upper, when

df_vbak_clean = (
    df_vbak
    .withColumn("WAERK", upper(trim(col("WAERK"))))
    .withColumn(
        "WAERK",
        when(col("WAERK").isNull(), "UNKNOWN")
        .otherwise(col("WAERK"))
    )
)

In [0]:
display(
    df_vbak_clean
    .select("WAERK")
    .distinct()
)

WAERK
EGP
UNKNOWN


In [0]:
from pyspark.sql.functions import col, trim, upper, when

df_vbak_clean = (
    df_vbak_clean
    .withColumn("GBSTK", upper(trim(col("GBSTK"))))
    .withColumn(
        "GBSTK",
        when(col("GBSTK") == "COMPLETE", "C")
        .when(col("GBSTK").isNull(), "UNKNOWN")
        .otherwise(col("GBSTK"))
    )
)

In [0]:
display(
    df_vbak_clean
    .select("GBSTK")
    .distinct()
    .orderBy("GBSTK")
)

GBSTK
A
B
C
UNKNOWN


In [0]:
from pyspark.sql.functions import trim, col

df_vbak_clean = (
    df_vbak_clean
    .withColumn("VKORG", trim(col("VKORG")))
)

In [0]:
display(
    df_vbak_clean
    .select("VKORG")
    .distinct()
    .orderBy("VKORG")
)

VKORG
1000.0
2000.0
3000.0


In [0]:
from pyspark.sql.functions import col, lpad

df_vbak_clean = (
    df_vbak_clean
    .withColumn(
        "KUNNR",
        lpad(col("KUNNR").cast("string"), 10, "0")
    )
)

In [0]:
from pyspark.sql.functions import when

df_vbak_clean = (
    df_vbak_clean
    .withColumn(
        "KUNNR_MISSING_FLAG",
        when(col("KUNNR").isNull(), 1).otherwise(0)
    )
)

In [0]:
display(
    df_vbak_clean
    .filter(col("KUNNR_MISSING_FLAG") == 1)
    .select("VBELN", "KUNNR", "KUNNR_MISSING_FLAG")
)

VBELN,KUNNR,KUNNR_MISSING_FLAG
5000000015,null,1
5000000033,null,1
5000000068,null,1
5000000085,null,1
5000000111,null,1
5000000112,null,1
5000000206,null,1
5000000224,null,1
5000000244,null,1
5000000270,null,1


In [0]:
from pyspark.sql.functions import col, coalesce, try_to_date, lit

df_vbak_clean = (
    df_vbak_clean
    .withColumn(
        "AUDAT_CLEAN",
        coalesce(
            try_to_date(col("AUDAT"), lit("yyyy-MM-dd")),
            try_to_date(col("AUDAT"), lit("dd/MM/yyyy")),
            try_to_date(col("AUDAT"), lit("yyyy/MM/dd"))
        )
    )
)

In [0]:
display(
    df_vbak_clean
    .filter(col("AUDAT_CLEAN").isNull())
    .select("VBELN", "AUDAT", "AUDAT_CLEAN")
)

VBELN,AUDAT,AUDAT_CLEAN
5000000006,null,null
5000000020,2026-13-05,null
5000000044,null,null
5000000073,null,null
5000000083,2026-13-05,null
5000000086,null,null
5000000228,null,null
5000000251,null,null
5000000307,null,null
5000000344,null,null


In [0]:
from pyspark.sql.functions import col, when

df_vbak_clean = (
    df_vbak_clean
    .withColumn(
        "AUDAT_INVALID_FLAG",
        when(col("AUDAT_CLEAN").isNull(), 1).otherwise(0)
    )
)

In [0]:
df_vbak_clean = (
    df_vbak_clean
    .drop("AUDAT")
    .withColumnRenamed("AUDAT_CLEAN", "AUDAT")
)

In [0]:
display(
    df_vbak_clean.select(
        "VBELN",
        "AUDAT",
        "AUDAT_INVALID_FLAG"
    )
)

VBELN,AUDAT,AUDAT_INVALID_FLAG
5000000001,2025-03-30,0
5000000002,2026-03-15,0
5000000003,2026-07-04,0
5000000004,2025-07-06,0
5000000005,2025-10-30,0
5000000006,null,1
5000000007,2026-06-22,0
5000000008,2025-12-08,0
5000000009,2026-05-15,0
5000000010,2026-06-20,0


In [0]:
display(
    
    df_vbak_clean
    .groupBy("MANDT", "VBELN")
    .count()
    .filter(col("count") > 1)
)

MANDT,VBELN,count
100,5000000031,2
100,5000000038,2
100,5000000091,2
100,5000000180,2
100,5000000258,2
100,5000000299,2
100,5000000311,2
100,5000000314,2
100,5000000321,2
100,5000000334,2


In [0]:
df_vbak_clean = (
    df_vbak_clean
    .dropDuplicates(["MANDT", "VBELN"])
)

In [0]:
display(
    df_vbak_clean
    .groupBy("MANDT", "VBELN")
    .count()
    .filter(col("count") > 1)
)

MANDT,VBELN,count


In [0]:
from pyspark.sql.functions import col, count, when

print("Rows:", df_vbak_clean.count())

# Nulls
df_vbak_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in [
        "VBELN",
        "VKORG",
        "KUNNR",
        "AUDAT",
        "WAERK",
        "GBSTK"
    ]
]).display()

Rows: 600


VBELN,VKORG,KUNNR,AUDAT,WAERK,GBSTK
0,0,18,18,0,0


In [0]:
# Check duplicates
display(
    df_vbak_clean
    .groupBy("MANDT", "VBELN")
    .count()
    .filter(col("count") > 1)
)

# Check cleaned categorical values
display(df_vbak_clean.select("VKORG").distinct().orderBy("VKORG"))
display(df_vbak_clean.select("WAERK").distinct().orderBy("WAERK"))
display(df_vbak_clean.select("GBSTK").distinct().orderBy("GBSTK"))

MANDT,VBELN,count


VKORG
1000.0
2000.0
3000.0


WAERK
EGP
UNKNOWN


GBSTK
A
B
C
UNKNOWN


In [0]:
df_vbak_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sap_sd_project.silver.vbak_clean")